In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna
import re

In [2]:
data = pd.read_csv("/kaggle/input/datasetlr5sppr/parkinsons.data")

display(data.head())
print(data.shape)

,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


(195, 24)


In [7]:
# Переименуем все признаки кроме целевой заранее в самом датафрейме
data_renamed = data.copy()

safe_columns = {}
for col in data_renamed.columns:
    # для целевой и name можно оставить как есть
    if col in ["status", "name"]:
        safe_columns[col] = col
        continue
    # оставляем только латинские буквы, цифры и _
    new_col = re.sub(r'[^A-Za-z0-9_]+', '_', col)
    safe_columns[col] = new_col

data_renamed = data_renamed.rename(columns=safe_columns)

print("Old columns:")
print(list(data.columns))
print("\nNew columns:")
print(list(data_renamed.columns))

Old columns:
['name', 'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']

New columns:
['name', 'MDVP_Fo_Hz_', 'MDVP_Fhi_Hz_', 'MDVP_Flo_Hz_', 'MDVP_Jitter_', 'MDVP_Jitter_Abs_', 'MDVP_RAP', 'MDVP_PPQ', 'Jitter_DDP', 'MDVP_Shimmer', 'MDVP_Shimmer_dB_', 'Shimmer_APQ3', 'Shimmer_APQ5', 'MDVP_APQ', 'Shimmer_DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']


In [8]:
target_col = "status"

X = data_renamed.drop(columns=[target_col, "name"])
y = data_renamed[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Пункт 1: обучение всех бустингов и таблица метрик

In [11]:
models = {
    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=4,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=0,
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary",
        random_state=42,
        verbose=-1
    )
}
use_scaler_for_tree_models = False

In [13]:
results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    if use_scaler_for_tree_models:
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    else:
        clf = model
    
    clf.fit(X_train, y_train)
    
    # Предсказания меток и вероятностей
    y_pred = clf.predict(X_test)
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_test)[:, 1]
    else:
        # Если нет predict_proba, можно использовать decision_function,
        # но для этих моделей predict_proba обычно есть.
        y_proba = None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    if y_proba is not None:
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan
    
    results.append({
        "model": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc_auc
    })

Training AdaBoost...
Training GradientBoosting...
Training XGBoost...
Training CatBoost...
Training LightGBM...


In [14]:
# Таблица результатов
results_df = pd.DataFrame(results).set_index("model")
display(results_df.sort_values("f1", ascending=False))

,accuracy,precision,recall,f1,roc_auc
model,,,,,
CatBoost,0.948718,0.965517,0.965517,0.965517,0.986207
XGBoost,0.923077,0.933333,0.965517,0.949153,0.982759
LightGBM,0.923077,0.933333,0.965517,0.949153,0.972414
GradientBoosting,0.923077,0.964286,0.931034,0.947368,0.975862
AdaBoost,0.846154,0.925926,0.862069,0.892857,0.951724


### Пункт 2: настройка XGBoost с Optuna

In [15]:
def objective(trial):
    # Гиперпараметры для XGBoost
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),# число деревьев
        "max_depth": trial.suggest_int("max_depth", 2, 10),# глубина деревьев
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),# скорость обучения
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),# доля строк на дерево
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),# доля столбцов
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),# минимальный прирост
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),# L1-регуляризация
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),# L2-регуляризация
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }    
    
    model = XGBClassifier(**params)
    
    # Кросс-валидация
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Будем максимизировать F1-меру
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )
    
    return scores.mean()

In [16]:
# Запуск Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2025-12-06 10:33:42,749] A new study created in memory with name: no-name-853ed5b9-ae78-455b-8fe6-598c805143c1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-06 10:33:45,140] Trial 0 finished with value: 0.8785759556347792 and parameters: {'n_estimators': 453, 'max_depth': 10, 'learning_rate': 0.16821756638476232, 'subsample': 0.5426929468631301, 'colsample_bytree': 0.8402233096944465, 'gamma': 4.516982876511196, 'reg_alpha': 3.4282679614129137, 'reg_lambda': 2.358065844392714, 'min_child_weight': 2.594380508354642}. Best is trial 0 with value: 0.8785759556347792.
[I 2025-12-06 10:33:45,419] Trial 1 finished with value: 0.881700380999194 and parameters: {'n_estimators': 509, 'max_depth': 4, 'learning_rate': 0.167374771713093, 'subsample': 0.5188341204716993, 'colsample_bytree': 0.6007457483285666, 'gamma': 4.064822999847061, 'reg_alpha': 0.06813533936979754, 'reg_lambda': 1.573790592520814, 'min_child_weight': 4.240231297906602}. Best is trial 1 with value: 0.881700380999194.
[I 2025-12-06 10:33:45,544] Trial 2 finished with value: 0.88178231292517 and parameters: {'n_estimators': 146, 'max_depth': 3, 'learning_rate': 0.250779249

In [17]:
print("Best trial:")
best_trial = study.best_trial
print("  F1 score:", best_trial.value)
print("  Params:")
for k, v in best_trial.params.items():
    print(f"    {k}: {v}")

Best trial:
  F1 score: 0.9304757809478306
  Params:
    n_estimators: 300
    max_depth: 8
    learning_rate: 0.05923822871459282
    subsample: 0.9044504944136816
    colsample_bytree: 0.559488872705447
    gamma: 0.8029271465868533
    reg_alpha: 1.0777059779917526
    reg_lambda: 2.408009038026212
    min_child_weight: 2.8318711674011334


Обучение финальной XGBoost‑модели с лучшими параметрами

In [18]:
best_params = study.best_params.copy()
best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
})

best_xgb = XGBClassifier(**best_params)
best_xgb.fit(X_train, y_train)

y_pred_opt = best_xgb.predict(X_test)
y_proba_opt = best_xgb.predict_proba(X_test)[:, 1]

acc_opt = accuracy_score(y_test, y_pred_opt)
prec_opt = precision_score(y_test, y_pred_opt)
rec_opt = recall_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)
roc_auc_opt = roc_auc_score(y_test, y_proba_opt)

print("XGBoost (Optuna-tuned) metrics on test:")
print(f"Accuracy:  {acc_opt:.4f}")
print(f"Precision: {prec_opt:.4f}")
print(f"Recall:    {rec_opt:.4f}")
print(f"F1:        {f1_opt:.4f}")
print(f"ROC AUC:   {roc_auc_opt:.4f}")

XGBoost (Optuna-tuned) metrics on test:
Accuracy:  0.9231
Precision: 0.9333
Recall:    0.9655
F1:        0.9492
ROC AUC:   0.9586


In [19]:
results_df.loc["XGBoost_Optuna"] = {
    "accuracy": acc_opt,
    "precision": prec_opt,
    "recall": rec_opt,
    "f1": f1_opt,
    "roc_auc": roc_auc_opt
}

display(results_df.sort_values("f1", ascending=False))

,accuracy,precision,recall,f1,roc_auc
model,,,,,
CatBoost,0.948718,0.965517,0.965517,0.965517,0.986207
XGBoost,0.923077,0.933333,0.965517,0.949153,0.982759
LightGBM,0.923077,0.933333,0.965517,0.949153,0.972414
XGBoost_Optuna,0.923077,0.933333,0.965517,0.949153,0.958621
GradientBoosting,0.923077,0.964286,0.931034,0.947368,0.975862
AdaBoost,0.846154,0.925926,0.862069,0.892857,0.951724
